# Шаг 2. Страницы 5–27514: расписание → DataFrame

Одна строка — правило расписания на период и дни недели. Это не отдельный вылет и не запись о пассажире. Полный разбор выполняется CLI один раз; notebook читает готовые Parquet. Две колонки обрабатываются независимо, повторы сохраняются.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "parse_intro.py").is_file():
    ROOT = ROOT / "parsers" / "PDF_Parser"
if not (ROOT / "parse_intro.py").is_file():
    raise FileNotFoundError(
        "Откройте notebook из корня проекта или папки parsers/PDF_Parserr"
    )
sys.path.insert(0, str(ROOT))

In [2]:
from parse_schedule import load_schedule

OUT = ROOT / "data/pdf_schedule"
manifest = json.loads((OUT / "manifest.json").read_text())
manifest

{'parser_version': 'skyteam.schedule.v1', 'source_file': 'Skyteam_Timetable.pdf', 'source_sha256': 'b7b3848d320e788518c315fd2d7655106dd172692238bb34e02c9e8cd4daf34d', 'coverage_start': '2018-11-01', 'coverage_end': '2019-01-31', 'status': 'complete', 'start_page': 5, 'end_page': 27514, 'unit': 'recurring schedule rule; not an individual or completed flight', 'seconds': 379.284, 'tables': {'raw_lines': 1856555, 'schedule': 1527643, 'quarantine': 0, 'operator_notes': 222288, 'pages': 55020}, 'counters': {'line_origin_header': 23648, 'line_destination_header': 23648, 'line_table_header': 47296, 'route_explicit_headers': 336209, 'equipment_listed_in_intro': 1436508, 'line_schedule_row': 1527643, 'row_like_lines': 1527643, 'route_continued_same_column': 1191434, 'no_details_notices': 12032, 'line_no_schedule_details_notice': 12032, 'equipment_not_listed_in_intro': 91135, 'line_operator_note': 222288}, 'field_issues': {'ambiguous_arrival_day_marker': 60}, 'verification': 'Technical parse onl

## Загрузка в pandas

По умолчанию выбираем нужные поля, чтобы не занимать память исходными текстами и служебными столбцами. Для всех 40 столбцов можно вызвать `load_schedule(OUT)` без columns.

In [3]:
columns = [
    "schedule_id",
    "page",
    "column",
    "origin_code",
    "destination_code",
    "valid_from",
    "valid_to",
    "days_codes",
    "departure_local",
    "arrival_local",
    "arrival_day_offset",
    "flight_raw",
    "aircraft_code",
    "duration_minutes",
    "route_status",
    "parse_status",
    "field_issues",
]
df = load_schedule(OUT, columns=columns)
print("Размер DataFrame:", df.shape)
df.head(10)

Размер DataFrame: (1527643, 17)


                                         schedule_id  ...  field_issues
0  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
1  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
2  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
3  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
4  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
5  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
6  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
7  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
8  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              
9  b7b3848d320e788518c315fd2d7655106dd172692238bb...  ...              

[10 rows x 17 columns]

`days_codes="134567"` означает все перечисленные дни, где 1 — воскресенье. Даты получили год только при единственном варианте внутри периода с обложки. Время остаётся местным; UTC и фактический оператор из номера рейса не выводятся.

In [4]:
df["route_status"].value_counts(dropna=False)

route_status
continued_same_column    1191434
explicit_headers          336209
Name: count, dtype: int64[pyarrow]

## Неполные и неоднозначные сведения

`explicit_headers`: оба заголовка напечатаны на той же странице; `continued_same_column`: использованы предыдущие заголовки этой колонки; `incomplete_route`: надёжно прочитать оба кода не удалось. Наследование — явно указанное правило парсера, а не независимое подтверждение маршрута.

In [5]:
df.loc[df["field_issues"] != "", ["page", "flight_raw", "field_issues"]].head(20)

         page flight_raw                  field_issues
811320  13094      CZ377  ambiguous_arrival_day_marker
811321  13094    MF9869*  ambiguous_arrival_day_marker
811322  13094      CZ377  ambiguous_arrival_day_marker
811323  13094    MF9869*  ambiguous_arrival_day_marker
811324  13094      CZ377  ambiguous_arrival_day_marker
811325  13094    MF9869*  ambiguous_arrival_day_marker
811330  13094      CZ377  ambiguous_arrival_day_marker
811331  13094    MF9869*  ambiguous_arrival_day_marker
811332  13094      CZ377  ambiguous_arrival_day_marker
811333  13094    MF9869*  ambiguous_arrival_day_marker
811334  13094      CZ377  ambiguous_arrival_day_marker
811335  13094    MF9869*  ambiguous_arrival_day_marker
811340  13094      CZ377  ambiguous_arrival_day_marker
811341  13094    MF9869*  ambiguous_arrival_day_marker
811342  13094      CZ377  ambiguous_arrival_day_marker
811343  13094    MF9869*  ambiguous_arrival_day_marker
811344  13094      CZ377  ambiguous_arrival_day_marker
811345  13

In [6]:
sample = load_schedule(OUT, filters=[("page", "=", 5)])
sample[
    [
        "origin_code",
        "destination_code",
        "validity_raw",
        "days_raw",
        "departure_local",
        "arrival_local",
        "flight_raw",
        "aircraft_code",
        "duration_raw",
    ]
].head(10)

  origin_code destination_code  ... aircraft_code duration_raw
0         AAL              AMS  ...           73W        1H25M
1         AAL              AMS  ...           73W        1H25M
2         AAL              AMS  ...           EQV        1H20M
3         AAL              AMS  ...           EQV        1H20M
4         AAL              AMS  ...           EQV        1H20M
5         AAL              AMS  ...           73W        1H25M
6         AAL              AMS  ...           73W        1H25M
7         AAL              AMS  ...           EQV        1H20M
8         AAL              AMS  ...           EQV        1H20M
9         AAL              AMS  ...           EQV        1H20M

[10 rows x 9 columns]

## Откуда взяты значения

`raw_id` связывает строку с исходным текстом, страницей, колонкой и номером текстовой строки. `origin_header_id` и `destination_header_id` указывают конкретные заголовки маршрута. Номер строки относится к тексту, извлечённому из колонки, а не к порядку рейсов в таблице. Нумерация страниц — с 1, колонок — 0/1.

In [7]:
raw_df = pd.read_parquet(OUT / "raw_lines.parquet", filters=[("page", "=", 5)])
raw_df[["page", "column", "line", "raw_text", "line_kind"]].head(12)

    page  ...           line_kind
0      5  ...       origin_header
1      5  ...  destination_header
2      5  ...        table_header
3      5  ...        table_header
4      5  ...        schedule_row
5      5  ...        schedule_row
6      5  ...        schedule_row
7      5  ...        schedule_row
8      5  ...        schedule_row
9      5  ...        schedule_row
10     5  ...        schedule_row
11     5  ...        schedule_row

[12 rows x 5 columns]

## Явные примечания об операторе

Это отдельная таблица, а не предположение по codeshare. Примечание связывается с предшествующей строкой в той же колонке, в том числе на границе страниц. Сохраняется статус связи. Звёздочка в номере рейса сохраняется отдельно без дополнительной интерпретации.

In [8]:
notes = pd.read_parquet(OUT / "operator_notes.parquet", filters=[("page", "=", 200)])
notes[["page", "column", "operator_raw", "link_status"]].head(10)

   page  column        operator_raw                  link_status
0   200       0  Qantas Airways Ltd  preceding_row_previous_page
1   200       0         V Australia      preceding_row_same_page
2   200       0         V Australia      preceding_row_same_page
3   200       0         V Australia      preceding_row_same_page
4   200       0         V Australia      preceding_row_same_page
5   200       0         V Australia      preceding_row_same_page
6   200       0  Qantas Airways Ltd      preceding_row_same_page
7   200       0         V Australia      preceding_row_same_page
8   200       0  Qantas Airways Ltd      preceding_row_same_page
9   200       0  Qantas Airways Ltd      preceding_row_same_page

## Контроль потерь и примеры

Сохраняются все текстовые строки, включая сообщения без подробного расписания. Карантин содержит то, что не удалось классифицировать. Нулевой карантин не означает отсутствия ошибок: неоднозначности полей хранятся в schedule.field_issues. Пример CSV содержит только первые 200 строк, полный результат — schedule.parquet.

In [9]:
quarantine = pd.read_parquet(OUT / "quarantine.parquet")
print("Карантин:", len(quarantine))
validation = json.loads((OUT / "validation.json").read_text())
validation

Карантин: 0


{'checks': {'source_sha256_matches': True, 'parquet_row_counts': True, 'page_column_coverage': True, 'page_sums': True, 'all_native_lines_accounted': True, 'row_like_conservation': True, 'unique_schedule_locators': True, 'header_lineage_same_column': True, 'normalized_periods_within_cover': True, 'identifier_string_types': True, 'assigned_operator_note_lineage': True, 'selected_pages_exact_text_reextraction': True}, 'raw_line_kinds': {'schedule_row': 1527643, 'operator_note': 222288, 'table_header': 47296, 'origin_header': 23648, 'destination_header': 23648, 'no_schedule_details_notice': 12032}, 'unassigned_operator_notes': 0, 'text_rechecked_pages': [5, 6, 100, 200, 1000, 13094, 15000, 27514], 'limitation': 'Technical integrity only; repeated source rows retained; inherited routes and source claims require separate review.'}

## Повторы

Повторяющиеся строки видны в самом PDF. На этапе парсинга не удаляем их, чтобы не потерять происхождение. Сравнение строк без учёта страницы не равно числу фактических рейсов. Независимая историческая проверка расписания не выполнена.